In [2]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor

from xrfm import xRFM

In [1]:
def prepare_tabular_data(df, target_col, drop_cols=None, test_size=0.2, val_size=0.2, random_state=42):
    if drop_cols is None:
        drop_cols = []

    # Separate X and y
    y = df[target_col].to_numpy()
    X = df.drop(columns=[target_col] + drop_cols)

    # Identify column types
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object"]).columns

    # Preprocessing pipelines
    num_pipeline = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]

    cat_pipeline = [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_pipeline), num_cols),
            ("cat", Pipeline(cat_pipeline), cat_cols)
        ]
    )

    # Split: train/test
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y if len(np.unique(y)) < 10 else None
    )

    # Split: train/val
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=val_size, random_state=random_state,
        stratify=y_train_full if len(np.unique(y)) < 10 else None
    )

    # Fit on train only
    X_train_proc = preprocessor.fit_transform(X_train)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    # Convert to dense (important for xRFM)
    X_train_proc = X_train_proc.toarray() if hasattr(X_train_proc, "toarray") else X_train_proc
    X_val_proc = X_val_proc.toarray() if hasattr(X_val_proc, "toarray") else X_val_proc
    X_test_proc = X_test_proc.toarray() if hasattr(X_test_proc, "toarray") else X_test_proc

    # Force numeric dtype
    X_train_proc = X_train_proc.astype("float32")
    X_val_proc = X_val_proc.astype("float32")
    X_test_proc = X_test_proc.astype("float32")

    return X_train_proc, X_val_proc, X_test_proc, y_train, y_val, y_test, preprocessor

In [3]:
hc_df = pd.read_csv("../datasets/healthcare-dataset-stroke-data.csv")
hc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [4]:
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = prepare_tabular_data(
    hc_df,
    target_col="stroke",
    drop_cols=["id"]
)

In [5]:
np.random.seed(0)
DEVICE = torch.device("cuda")
bw = 5.
reg = 1e-3
iters = 5
max_leaf_size = 55_000

xrfm_params = {
    'model': {
        'kernel': "sum_power_laplace",
        'bandwidth': bw,
        'exponent': 1.0,
        'diag': False,
        'bandwidth_mode': "constant"
    },
    'fit': {
        'reg': reg,
        'iters': iters,
        'verbose': True,
        'early_stop_rfm': True,
    }
}
default_rfm_params = {
    'model': {
        "kernel": 'l2_high_dim',
        "exponent": 1.0,
        "bandwidth": 10.0,
        "diag": False,
        "bandwidth_mode": "constant"
    },
    'fit' : {
        "get_agop_best_model": True,
        "return_best_params": False,
        "reg": 1e-3,
        "iters": 0,
        "early_stop_rfm": False,
        "verbose": False
    }
}
xrfm_model = xRFM(
    xrfm_params,
    device="cpu",
    tuning_metric="accuracy",
    max_leaf_size=max_leaf_size,
    default_rfm_params=default_rfm_params,
    split_method="top_vector_agop_on_subset"
)

xrfm_model.fit(X_train, y_train, X_val, y_val)
y_pred = xrfm_model.predict(X_test)

{'model': {'kernel': 'l2_high_dim', 'exponent': 1.0, 'bandwidth': 10.0, 'diag': False, 'bandwidth_mode': 'constant'}, 'fit': {'get_agop_best_model': True, 'return_best_params': False, 'reg': 0.001, 'iters': 0, 'early_stop_rfm': False, 'verbose': False}}
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 3270, d: 21, and nval: 818
Round 0 Val accuracy: 0.9523
Using cheap batch size
Optimal M batch size: 3270
Sampling AGOP on maximum of 22890 total points


  0%|          | 0/1 [00:00<?, ?it/s]

Using SVD
Time taken for round 0: 1.5798351764678955 seconds
Round 1 Val accuracy: 0.9511
Using cheap batch size
Optimal M batch size: 3270
Sampling AGOP on maximum of 22890 total points


...

Using SVD
Time taken for round 3: 0.40285587310791016 seconds
Round 4 Val accuracy: 0.9511
Using cheap batch size
Optimal M batch size: 3270
Sampling AGOP on maximum of 22890 total points


  0%|          | 0/1 [00:00<?, ?it/s]

Using SVD
Time taken for round 4: 0.3991701602935791 seconds


Building trees:   0%|          | 0/1 [00:03<?, ?it/s]

Final Val accuracy: 0.9511
self.best_iter=0
Tree has no split, stopping training
Using hard routing for tree prediction


In [8]:
accuracy = np.sum(y_pred == y_test)/len(y_test)
print(accuracy)

0.9442270058708415
